In [1]:
# ! pip install healpy numpyro corner arviz optax reproject einops seaborn chainconsumer blackjax

In [2]:
! nvidia-smi

Wed Dec 21 17:15:23 2022       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 515.65.01    Driver Version: 515.65.01    CUDA Version: 11.7     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  Tesla V100-PCIE...  On   | 00000000:06:00.0 Off |                    0 |
| N/A   38C    P0    25W / 250W |      0MiB / 32768MiB |      0%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
                                                                               
+-------

In [2]:
import sys
import pickle
sys.path.append("../")

from jax.config import config
config.update("jax_enable_x64", True)

import jax
import jax.numpy as jnp
import numpy as np
import corner
import matplotlib.pyplot as plt

from models.np_model import NPModel

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
r_outer = 25
l_max = 0

vary_disk = 0
vary_gamma = 1
bulge_hybrid = 1

dif = "ModelO"
ps_cat = "3fgl"
nside = 128
n_exp = 1

npmodel = NPModel(dif=dif, r_outer=r_outer, l_max=l_max, vary_disk=vary_disk, vary_gamma=vary_gamma, bulge_hybrid=bulge_hybrid, ps_cat=ps_cat, nside=nside, n_exp=n_exp)

Loading the psf correction from: /net/rcstorenfs02/ifs/rc_labs/dvorkin_lab/smsharma/mi-attribution/notebooks/psf_dir/Fermi_PSF_2GeV2_nside128.npy
Max photon count is 103


In [21]:
svi_results = npmodel.fit_svi(rng_key=jax.random.PRNGKey(4234), n_steps=5000, lr=1e-2)

In [12]:
arviz_post = npmodel.get_posterior_samples(rng_key=jax.random.PRNGKey(42342), num_samples=50000)
posterior = arviz_post.to_dict()['posterior']

In [13]:
import arviz as az
az.summary(posterior)

Shape validation failed: input_shape: (1, 50000), minimum_shape: (chains=2, draws=4)


,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
S_bub,0.363,0.345,0.013,0.983,0.002,0.001,49853.0,48679.0,NaN
S_dif,12.168,0.552,11.119,13.142,0.003,0.002,48351.0,48573.0,NaN
S_gce,0.572,0.371,0.037,1.269,0.002,0.001,49986.0,48227.0,NaN
S_ics,4.115,0.748,2.664,4.976,0.003,0.002,49222.0,49298.0,NaN
S_iso,2.809,0.929,1.099,4.416,0.004,0.003,49721.0,49379.0,NaN
S_psc,2.564,1.517,0.271,4.964,0.007,0.005,49150.0,49226.0,NaN
Sps_dsk,1.424,0.416,0.638,1.986,0.002,0.001,50178.0,49095.0,NaN
Sps_gce,0.954,0.358,0.319,1.603,0.002,0.001,50110.0,49442.0,NaN
lambdas_dsk,0.551,0.241,0.151,0.939,0.001,0.001,48503.0,48841.0,NaN
lambdas_gce,0.549,0.252,0.146,0.957,0.001,0.001,50605.0,49710.0,NaN


In [82]:
from likelihoods.npll_jax import log_like_np

In [83]:
exp_lens = [len(npmodel.expreg_indices[i]) for i in range(len(npmodel.expreg_indices))]
n_pad = exp_lens[0] - exp_lens[-1]
npmodel.expreg_indices[-1] = jnp.pad(npmodel.expreg_indices[-1], (0, n_pad))

In [86]:
jnp.pad(npmodel.expreg_indices[-1], (0, n_pad)).shape

(571,)

In [88]:
[len(npmodel.expreg_indices[i]) for i in range(len(npmodel.expreg_indices))]

[570, 570, 570, 570, 570, 570, 570, 570, 570, 570, 570, 570]

In [89]:
npix = np.sum([len(npmodel.expreg_indices[i]) for i in range(n_exp)])
theta = jnp.ones((2, 6))
mu = jnp.ones(npix)
npt_compressed = jnp.ones((2, npix))
data = jnp.array((3 * np.ones(npix)).astype(np.int32))

In [90]:
mu_batch = mu[jnp.array(npmodel.expreg_indices)]
npt_compressed_batch = npt_compressed[:, jnp.array(npmodel.expreg_indices)]
data_batch = data[jnp.array(npmodel.expreg_indices)]

In [91]:
log_like_np(theta, mu[npmodel.expreg_indices[0]], npt_compressed[:, npmodel.expreg_indices[0]], data[npmodel.expreg_indices[0]], npmodel.f_ary, npmodel.df_rho_div_f_ary, npmodel.k_max, len(npmodel.expreg_indices[0]));

In [93]:
log_like_exp = jax.vmap(log_like_np, in_axes=(None,0,1,0,None,None,None,None))(theta, mu_batch, npt_compressed_batch, data_batch, npmodel.f_ary, npmodel.df_rho_div_f_ary, npmodel.k_max, len(npmodel.expreg_indices[0]))
log_like_exp.at[-1, -n_pad:].set(0.)

DeviceArray([[-40.99357513, -40.99357513, -40.99357513, ...,
              -40.99357513, -40.99357513, -40.99357513],
             [-40.99357513, -40.99357513, -40.99357513, ...,
              -40.99357513, -40.99357513, -40.99357513],
             [-40.99357513, -40.99357513, -40.99357513, ...,
              -40.99357513, -40.99357513, -40.99357513],
             ...,
             [-40.99357513, -40.99357513, -40.99357513, ...,
              -40.99357513, -40.99357513, -40.99357513],
             [-40.99357513, -40.99357513, -40.99357513, ...,
              -40.99357513, -40.99357513, -40.99357513],
             [-40.99357513, -40.99357513, -40.99357513, ...,
              -40.99357513, -40.99357513,   0.        ]], dtype=float64)